# B2-020-language-transformers — Practice p20 — Solution

**Type:** integrative · **Difficulty:** advanced · **Concepts:** nlp-fine-tuning-protocol, transformer-nlp-task-design

*65 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260812`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C11-neural-training`, `B2-019-attention-transformers`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb).

## Pinned reproducibility protocol

Load only the hash-checked state and the literal intent rows/splits from `data/language_fixture.py`; seed the new two-class head with `20260812`.  Use AdamW with `lr=0.03`, `weight_decay=0`, betas `(0.9,0.999)`, eps `1e-8`, stored ascending order, and exactly 40 full-batch updates.  Do not shuffle, use an external corpus, or replace the loaded encoder.

## Solution

The encoder is reconstructed with the declared architecture and filled only from the hash-pinned student state. A separately reconstructed loaded encoder and identically seeded fresh head provide the untrained-head held-out baseline.

In [ ]:
import importlib.util

def load_literal_module(name, relative_path):
    spec = importlib.util.spec_from_file_location(name, relative_path)
    assert spec is not None and spec.loader is not None
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)

class TinyEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(12, 8, padding_idx=0)
        self.position_embedding = nn.Embedding(8, 8)
        self.norm1 = nn.LayerNorm(8, eps=1e-5)
        self.attention = nn.MultiheadAttention(8, 2, dropout=0.0, batch_first=True)
        self.norm2 = nn.LayerNorm(8, eps=1e-5)
        self.ff1 = nn.Linear(8, 16)
        self.ff2 = nn.Linear(16, 8)

    def forward(self, token_ids, *, mask_mode):
        length = token_ids.shape[1]
        positions = torch.arange(length, device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        normalized = self.norm1(x)
        attention_mask = None
        if mask_mode == "causal":
            attention_mask = torch.triu(
                torch.ones(length, length, dtype=torch.bool, device=token_ids.device),
                diagonal=1,
            )
        elif mask_mode != "bidirectional":
            raise ValueError(f"unknown mask mode: {mask_mode}")
        attended, _ = self.attention(
            normalized,
            normalized,
            normalized,
            attn_mask=attention_mask,
            key_padding_mask=token_ids.eq(0),
            need_weights=False,
        )
        x = x + attended
        return x + self.ff2(F.gelu(self.ff1(self.norm2(x))))

fixture = load_literal_module("language_fixture_p20", "../data/language_fixture.py")
state = load_literal_module("tiny_encoder_state_p20", "../data/tiny_encoder_state.py")
EXPECTED_STATE_HASH = "fc6b206ed1e55a8efb91c0ea5d821df8ff4a118ccafd5637c94be76c9a7f53a9"

def loaded_encoder():
    assert state.ENCODER_STATE_HASH == EXPECTED_STATE_HASH
    encoder = TinyEncoder()
    tensor_state = {name: torch.tensor(value, dtype=torch.float32) for name, value in state.ENCODER_TENSORS.items()}
    encoder.load_state_dict(tensor_state, strict=True)
    encoder.loaded_state_verified = all(torch.equal(value, tensor_state[name]) for name, value in encoder.state_dict().items())
    return encoder

def pooled_hidden(encoder, rows):
    hidden = encoder(rows, mask_mode="bidirectional")
    valid = rows.ne(0)
    return (hidden * valid.unsqueeze(-1)).sum(1) / valid.sum(1, keepdim=True)

def split_metrics(encoder, head, inputs, labels, split):
    logits = head(pooled_hidden(encoder, inputs[split]))
    return {
        "loss": F.cross_entropy(logits, labels[split], reduction="mean").item(),
        "accuracy": logits.argmax(1).eq(labels[split]).float().mean().item(),
        "shape": tuple(logits.shape),
    }

inputs = torch.tensor(fixture.INTENT_INPUT_IDS, dtype=torch.int64)
labels = torch.tensor(fixture.INTENT_LABELS, dtype=torch.int64)
splits = {"train": torch.tensor(fixture.TRAIN_SPLIT_IDS), "validation": torch.tensor(fixture.VALID_SPLIT_IDS), "test": torch.tensor(fixture.TEST_SPLIT_IDS)}
baseline_encoder = loaded_encoder()
torch.manual_seed(20260812)
baseline_head = nn.Linear(8, 2)
baseline = {name: split_metrics(baseline_encoder, baseline_head, inputs, labels, split) for name, split in splits.items()}

encoder = loaded_encoder()
torch.manual_seed(20260812)
head = nn.Linear(8, 2)
initial_state_verified = encoder.loaded_state_verified
optimizer = torch.optim.AdamW([*encoder.parameters(), *head.parameters()], lr=0.03, weight_decay=0, betas=(0.9,0.999), eps=1e-8)
optimized_ids = {id(parameter) for group in optimizer.param_groups for parameter in group["params"]}
for _ in range(40):
    optimizer.zero_grad(set_to_none=True)
    logits = head(pooled_hidden(encoder, inputs[splits["train"]]))
    loss = F.cross_entropy(logits, labels[splits["train"]], reduction="mean")
    loss.backward(); optimizer.step()
metrics = {name: split_metrics(encoder, head, inputs, labels, split) for name, split in splits.items()}
RESULT = (encoder, head, metrics)

### Answer check

In [ ]:
assert state.ENCODER_STATE_HASH == EXPECTED_STATE_HASH and initial_state_verified
assert set(fixture.TRAIN_SPLIT_IDS).isdisjoint(fixture.VALID_SPLIT_IDS)
assert set(fixture.TRAIN_SPLIT_IDS).isdisjoint(fixture.TEST_SPLIT_IDS)
assert set(fixture.VALID_SPLIT_IDS).isdisjoint(fixture.TEST_SPLIT_IDS)
assert optimized_ids == {id(parameter) for parameter in [*encoder.parameters(), *head.parameters()]}
assert pooled_hidden(encoder, inputs).shape == (6, 8)
assert all(metrics[name]["shape"] == (len(splits[name]), 2) for name in splits)
assert metrics["validation"]["loss"] < baseline["validation"]["loss"]
assert metrics["test"]["loss"] < baseline["test"]["loss"]
assert all(0.0 <= metrics[name]["accuracy"] <= 1.0 for name in metrics)
assert next(encoder.parameters()).dtype == head.weight.dtype == torch.float32